# IMPLEMENTATION OF TCN-BiLSTM ARCHITECTURE ON XJTU DATASET

This notebook demonstrates the training and validation of the TCN-BiLSTM-Attention architecture across multiple window sizes on the XJTU-SY bearing dataset.

In [1]:
import os
import glob
import time
import gc
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import Custom Architecture
import sys
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from importlib import import_module
tcn_bilstm_module = import_module('TCN-BiLSTM_Implementation')
TCN_BiLSTM_Attention = tcn_bilstm_module.TCN_BiLSTM_Attention

# Matplotlib Publication Standards
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'lines.linewidth': 2.0,
    'grid.alpha': 0.3
})
sns.set_style("whitegrid")


## 1. Global Configurations
Setup paths, experiment variables, and model hyperparameters.

In [2]:
# Dataset Paths
DATASET_DIR = r"D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset"
RESULTS_DIR = r"D:\Proyek Dosen\Riset Bearing\TCN_BiLSTM_Results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Experiment Setup
WINDOW_SIZES = [30, 40, 50, 70]
BEARING_LIFESPAN_TIME = 392_275 # Maximum lifespan for scaling RUL if needed
NUM_FEATURES = 15 # Based on sliding window pipeline

# Training Hyperparameters
EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_HEADS = 4

print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")


Num GPUs Available: 0


## 2. Helper Functions
Metrics, plotting, and evaluation utilities.

In [3]:
def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def relative_prediction_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def health_index_to_rul(hi, max_rul=BEARING_LIFESPAN_TIME):
    return np.clip(hi, 0, 1) * max_rul

def asymmetric_loss_scoring_function(y_pred, y_true, a=10, b=13):
    y_pred = tf.convert_to_tensor(y_pred, dtype=tf.float32)
    y_true = tf.convert_to_tensor(y_true, dtype=tf.float32)
    diff = y_pred - y_true
    
    loss_early = tf.exp(-diff / a) - 1.0
    loss_late = tf.exp(diff / b) - 1.0
    loss = tf.where(diff < 0, loss_early, loss_late)
    return tf.reduce_mean(loss).numpy()

def plot_health_index_prediction(y_true, y_pred, title, save_path=None):
    plt.figure(figsize=(12, 6))
    plt.plot(y_true, label='True Health Index', color='black', alpha=0.8)
    plt.plot(y_pred, label='Predicted Health Index', color='gold', linestyle='--')
    plt.title(title, fontweight='bold')
    plt.xlabel('Time Steps')
    plt.ylabel('Health Index (0: Failed, 1: Normal)')
    plt.legend(loc='upper right')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


## 3. Training & Evaluation Pipeline
Looping over specified window sizes to train models and evaluate on validation bearings.

In [ ]:
# Summary Storage
summary_metrics = []
# For saving predictions across all experiments into an Excel file
excel_writer_path = os.path.join(RESULTS_DIR, "TCN_BiLSTM_Predictions_Summary.xlsx")
excel_predictions = {} 

for ws in WINDOW_SIZES:
    print(f"\n{'='*60}\nStarting Experiment for Window Size: {ws}\n{'='*60}")
    
    ws_res_dir = os.path.join(RESULTS_DIR, f"window_size_{ws}")
    os.makedirs(ws_res_dir, exist_ok=True)
    
    # ----------------------------------------------------
    # A. LOAD TRAINING DATA
    # ----------------------------------------------------
    train_file = os.path.join(DATASET_DIR, f"processed_train_w{ws}.csv")
    if not os.path.exists(train_file):
        print(f"WARNING: Train file {train_file} not found. Skipping...")
        continue
        
    print(f"Loading training data: {train_file}")
    df_train = pd.read_csv(train_file)
    
    if df_train.empty:
        print(f"WARNING: Training dataset for Window Size {ws} is empty. Skipping...")
        continue
    
    y_train = df_train['Target_RUL'].values
    
    # Exclude metadata columns
    drop_cols = ['Target_RUL', 'Bearing_ID'] if 'Bearing_ID' in df_train.columns else ['Target_RUL']
    X_train_flat = df_train.drop(columns=drop_cols).values
    
    num_samples = X_train_flat.shape[0]
    num_features = X_train_flat.shape[1] // ws
    X_train_3d = X_train_flat.reshape(num_samples, ws, num_features)
    
    # Free memory
    del df_train, X_train_flat
    gc.collect()
    
    # ----------------------------------------------------
    # B. INITIALIZE MODEL & TRAIN (TCN-BiLSTM)
    # ----------------------------------------------------
    tf.keras.backend.clear_session()
    
    model_wrapper = TCN_BiLSTM_Attention(
        window_size=ws,
        num_features=num_features,
        num_heads=NUM_HEADS
    )
    
    # Compile with parameters matching paper / previous config
    # The class sets learning_rate=0.001 internally but we can recompile if we want explicitly
    
    print("Training Model...")
    
    # Using Early Stopping Callback
    early_stop_cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )
    
    # We use validation_split=0.2 on the train set directly representing tuning
    history = model_wrapper.model.fit(
        X_train_3d, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stop_cb],
        verbose=0  # set to 1 or 2 for detailed output
    )
    
    model_wrapper.model.save(os.path.join(ws_res_dir, "best_tcn_bilstm_model.h5"))
    
    # Plot Training Loss
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Train MSE Loss', color='red')
    plt.plot(history.history['val_loss'], label='Val MSE Loss', color='blue')
    plt.title(f'Training Loss History (Window Size = {ws})')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ws_res_dir, "training_loss_history.png"), dpi=300)
    plt.close()
    
    # Free Memory
    del X_train_3d, y_train
    gc.collect()
    tf.keras.backend.clear_session()
    
    # ----------------------------------------------------
    # C. EVALUATION ON VALIDATION BEARINGS
    # ----------------------------------------------------
    val_files = glob.glob(os.path.join(DATASET_DIR, f"processed_val_*_w{ws}.csv"))
    
    ws_predictions_df = pd.DataFrame()
    ws_metrics = []
    
    for v_file in val_files:
        b_match = re.search(r"processed_val_(.*)_w\d+\.csv", os.path.basename(v_file))
        b_id = b_match.group(1) if b_match else 'Unknown'
        print(f"Evaluating on {b_id} (Window={ws})...")
        
        df_val = pd.read_csv(v_file)
        
        if df_val.empty:
            print(f"WARNING: Validation dataset for {b_id} (Window Size {ws}) is empty. Skipping metric calculation...")
            continue
            
        y_val_true = df_val['Target_RUL'].values
        val_drop_cols = ['Target_RUL', 'Bearing_ID'] if 'Bearing_ID' in df_val.columns else ['Target_RUL']
        X_val_flat = df_val.drop(columns=val_drop_cols).values
        
        ns = X_val_flat.shape[0]
        nf = X_val_flat.shape[1] // ws
        X_val_3d = X_val_flat.reshape(ns, ws, nf)
        
        # Predict
        y_val_pred = model_wrapper.model.predict(X_val_3d, batch_size=1024, verbose=0).flatten()
        
        # Metrics
        rmse_val = rmse_score(y_val_true, y_val_pred)
        mae_val = mean_absolute_error(y_val_true, y_val_pred)
        r2_val = r2_score(y_val_true, y_val_pred)
        rpe_val = relative_prediction_error(y_val_true, y_val_pred)
        asym_loss = asymmetric_loss_scoring_function(y_val_pred, y_val_true)
        
        ws_metrics.append({
            'Bearing_ID': b_id,
            'Window_Size': ws,
            'RMSE': rmse_val,
            'MAE': mae_val,
            'R2': r2_val,
            'RPE(%)': rpe_val,
            'Asymmetric_Loss': asym_loss
        })
        
        # DataFrame
        b_pred_df = pd.DataFrame({
            'Bearing_ID': b_id,
            'Time_Step': np.arange(len(y_val_true)),
            'True_Health_Index': y_val_true,
            'Predicted_Health_Index': y_val_pred,
            'True_RUL': health_index_to_rul(y_val_true),
            'Predicted_RUL': health_index_to_rul(y_val_pred)
        })
        ws_predictions_df = pd.concat([ws_predictions_df, b_pred_df], ignore_index=True)
        
        # Plot
        plot_title = f"{b_id} Health Index Prediction (TCN-BiLSTM - WS={ws})"
        plot_save_path = os.path.join(ws_res_dir, f"prediction_plot_{b_id}.png")
        plot_health_index_prediction(y_val_true, y_val_pred, plot_title, plot_save_path)
        
        del df_val, X_val_flat, X_val_3d
        gc.collect()
            
    # Save WS Predictions
    if not ws_predictions_df.empty:
        ws_predictions_df.to_csv(os.path.join(ws_res_dir, f"predictions_ws_{ws}.csv"), index=False)
        excel_predictions[f"WS_{ws}"] = ws_predictions_df
    
    # Store metrics for Window Size
    if len(ws_metrics) > 0:
        ws_metrics_df = pd.DataFrame(ws_metrics)
        ws_metrics_df.to_csv(os.path.join(ws_res_dir, f"metrics_ws_{ws}.csv"), index=False)
        mean_metrics = ws_metrics_df.mean(numeric_only=True)
        
        summary_metrics.append({
            'Window_Size': ws,
            'Mean_RMSE': mean_metrics['RMSE'],
            'Mean_MAE': mean_metrics['MAE'],
            'Mean_R2': mean_metrics['R2'],
            'Mean_RPE(%)': mean_metrics['RPE(%)'],
            'Mean_Asymmetric_Loss': mean_metrics['Asymmetric_Loss']
        })

# Export Combined Predictions
if excel_predictions:
    print(f"\nSaving all predictions to summary Excel: {excel_writer_path}")
    with pd.ExcelWriter(excel_writer_path) as writer:
        for sheet_name, df in excel_predictions.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print("All Experiments Completed Succesfully!")
else:
    print("WARNING: No valid prediction matrices generated due to potentially missing datasets.")


Starting Experiment for Window Size: 30
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w30.csv

Training Model...
Restoring model weights from the end of the best epoch: 200.



Starting Experiment for Window Size: 40
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w40.csv
Training Model...
Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 1.



Starting Experiment for Window Size: 50
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w50.csv
Training Model...
Epoch 112: early stopping
Restoring model weights from the end of the best epoch: 97.



Starting Experiment for Window Size: 70
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w70.csv
Training Model...
Restoring model weights from the end of the best epoch: 196.



Saving all predictions to summary Excel: D:\Proyek Dosen\Riset Bearing\TCN_BiLSTM_Results\TCN_BiLSTM_Predictions_Summary.xlsx
All Experiments Completed Succesfully!


## 4. Final Window Size Optimization Analysis
Comparing validating mean MSE/RMSE across the 4 window size variants to deduce the optimal length.

In [5]:
summary_df = pd.DataFrame(summary_metrics)

if not summary_df.empty:
    display(summary_df)
    summary_df.to_csv(os.path.join(RESULTS_DIR, "TCN_BiLSTM_WindowSize_Comparison.csv"), index=False)
    
    plt.figure(figsize=(10, 6))
    plt.plot(summary_df['Window_Size'], summary_df['Mean_RMSE'], marker='o', linestyle='-', color='purple', linewidth=2.5, markersize=8)
    
    # Annotate points
    for i, row in summary_df.iterrows():
        plt.annotate(f"{row['Mean_RMSE']:.4f}", 
                     (row['Window_Size'], row['Mean_RMSE']),
                     textcoords="offset points", 
                     xytext=(0,10), 
                     ha='center')
                     
    plt.title('Mean Validation RMSE vs Window Size (TCN-BiLSTM)', fontweight='bold', pad=15)
    plt.xlabel('Window Size')
    plt.ylabel('Mean Validation RMSE')
    plt.xticks(summary_df['Window_Size'].values)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "optim_tcn_bilstm_rmse_vs_windowsize.png"), dpi=300)
    plt.show()
    
    optimal_ws = summary_df.loc[summary_df['Mean_RMSE'].idxmin()]['Window_Size']
    print(f"\n>>> Based on Mean Validation RMSE, the Optimal Window Size is: {int(optimal_ws)} <<<")
else:
    print("No summary metrics available to analyze.")


No summary metrics available to analyze.
